# Fitting SANS data with a sphere

Small-angle neutron scattering (SANS) measures how a sample scatters neutrons at small angles. The shape of
the curve $I(q)$ carries the size and shape of the objects in the sample. This notebook fits a model of
spheres to reduced SANS data, and then goes one step further: it works out physical quantities that follow
from the fitted radius, with their uncertainty, using the arithmetic and the functions of `es.math`.

Each part closes with a few exercises on what was just shown. Their hints and solutions are folded away and
open on a click, so the text reads straight through; a solution is a worked answer with its result.

In [1]:
import numpy as np
import plotly.graph_objects as go
import pooch

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Fitter
from easyscience import Parameter
from easyscience.base_classes import ModelBase

The function `show` prints a labelled number as value, uncertainty and unit.

In [2]:
def show(label, number):
    """Print a labelled number as value ± uncertainty with its unit."""
    print(f'{label} = {number.value:.4g} ± {number.error or 0.0:.2g} {number.unit}')

## The data

The data are simulated and reduced SANS intensities $I(q)$ with their uncertainties, for the scattering vector
$q$ in $\text{Å}^{-1}$. They are stored in a plain text file with three columns, which is downloaded once
and then read from the local copy:

In [3]:
filename = pooch.retrieve(
    url='https://public.esss.dk/groups/scipp/dmsc-summer-school/2025/4-reduction/sans_iofq.dat',
    known_hash='sha256:fb29bc63f793cac8cb2b9832afa159469b8525cd39a1e134efef6357be4be67b',
)
q, intensity, error = np.loadtxt(filename, unpack=True)
keep = np.isfinite(intensity)
q, intensity, error = q[keep], intensity[keep], error[keep]

print(f'{q.size} points, q from {q.min():.4f} to {q.max():.3f} 1/Å')

200 points, q from 0.0055 to 0.190 1/Å


The chart is interactive: hover over a point to read its values. It loads Plotly from the internet, so it
needs a connection. The charts are shown with a small helper, `show_figure(fig)`, which is defined in a
hidden cell; in your own notebook `fig.show()` does the same.

In [4]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

In [5]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=q,
        y=intensity,
        mode='markers',
        name='data',
        error_y=dict(type='data', array=error, visible=True),
        marker=dict(size=4),
    )
)
fig.update_layout(
    xaxis_title='q / (1/Å)',
    yaxis_title='I(q)',
    yaxis_type='log',
    margin=dict(l=60, r=20, t=30, b=50),
)
show_figure(fig)

## The model

For a dilute sample of identical spheres of radius $r$ the intensity is

$$
I(q) = \frac{\text{scale}}{V}\left(\frac{3V\,\Delta\rho\,[\sin(qr) - qr\cos(qr)]}{(qr)^3}\right)^2 + \text{bkg},
\qquad V = \frac{4}{3}\pi r^3,
$$

where $\Delta\rho$ is the contrast between the spheres and the solvent and $\text{bkg}$ is a flat
background. The scale factor is known from the instrument and stays fixed, so the fit has three free
parameters: $\Delta\rho$, $r$ and $\text{bkg}$.

A model is a class that holds its parameters as `Parameter`s and is called with $q$. The parameters carry
units, so that results derived from them have units too. Inside `__call__` the model works with plain NumPy
arrays and the `.value` of each parameter: `es.math` takes one `Parameter` at a time, so it belongs
**after** the fit, on the fitted parameters.

In [6]:
class Sphere(ModelBase):
    """The scattering of identical spheres, as a function of q."""

    def __init__(self, scale: float, delta_rho: float, r: float, bkg: float):
        super().__init__()
        self._scale = Parameter(name='scale', value=scale, fixed=True)
        self._delta_rho = Parameter(
            name='delta_rho', value=delta_rho, unit='1/angstrom**2', min=0, max=10
        )
        self._r = Parameter(name='r', value=r, unit='angstrom', min=10, max=1000)
        self._bkg = Parameter(name='bkg', value=bkg, min=1e-3, max=1e-1)

    @property
    def scale(self) -> Parameter:
        return self._scale

    @property
    def delta_rho(self) -> Parameter:
        return self._delta_rho

    @property
    def r(self) -> Parameter:
        return self._r

    @property
    def bkg(self) -> Parameter:
        return self._bkg

    def __call__(self, q):
        qr = q * self._r.value
        volume = 4 / 3 * np.pi * self._r.value**3
        form_factor = 3 * volume * self._delta_rho.value * (np.sin(qr) - qr * np.cos(qr)) / qr**3
        return self._scale.value / volume * form_factor**2 + self._bkg.value

## The fit

The fit starts from a rough guess. The weights are the inverse uncertainties of the points. After the fit each parameter
holds its fitted value and its uncertainty:

In [7]:
model = Sphere(scale=1.4e-7, delta_rho=3.0, r=80.0, bkg=0.01)
fitter = Fitter(model, model)
result = fitter.fit(x=q, y=intensity, weights=1 / error)

print(f'converged: {result.success}, reduced chi-squared: {result.reduced_chi2:.2f}')
show('delta_rho', model.delta_rho)
show('r', model.r)
show('bkg', model.bkg)

converged: True, reduced chi-squared: 3.77
delta_rho = 3.521 ± 0.024 1/Å^2
r = 90.62 ± 0.26 Å
bkg = 0.01861 ± 0.00036 dimensionless


The reduced chi-squared is well above 1, so the sphere does not describe every detail of the data. It does
capture the overall shape, as the chart shows, and it is a sensible basis for the quantities below.

In [8]:
fit_q = np.linspace(q.min(), q.max(), 400)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=q,
        y=intensity,
        mode='markers',
        name='data',
        error_y=dict(type='data', array=error, visible=True),
        marker=dict(size=4),
    )
)
fig.add_trace(
    go.Scatter(x=fit_q, y=model(fit_q), mode='lines', name='sphere fit', line=dict(color='black'))
)
fig.update_layout(
    xaxis_title='q / (1/Å)',
    yaxis_title='I(q)',
    yaxis_type='log',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.65, y=0.95),
)
show_figure(fig)

## Size and volume

The fitted radius is a `Parameter` with the unit $\text{Å}$ and an uncertainty. Ordinary arithmetic gives the
diameter and the volume, and `convert_unit` expresses them in nanometres. The uncertainty is carried along
and converted with the value:

In [9]:
diameter = 2 * model.r
diameter.convert_unit('nm')
show('diameter', diameter)

volume = 4 / 3 * np.pi * model.r**3
volume.convert_unit('nm**3')
show('volume', volume)

diameter = 18.12 ± 0.053 nm
volume = 3117 ± 27 nm^3


### Exercise 1: Surface of the sphere

The surface area of a sphere is $4\pi r^2$. Work out the surface area of the fitted sphere in $\text{nm}^2$, with its uncertainty.

In [10]:
radius = model.r

<details class="question">
<summary>Hint 1</summary>

A plain number such as `np.pi` can multiply a `Parameter`, and `**` takes a plain-number exponent.

</details>

<details class="question">
<summary>Hint 2</summary>

`convert_unit('nm**2')` converts the result from $\text{Å}^2$ to $\text{nm}^2$, with its uncertainty.

</details>

In [11]:
%%solution The relative uncertainty of the surface is twice that of the radius.
surface = 4 * np.pi * radius**2
surface.convert_unit('nm**2')
show('surface', surface)

## The radius of gyration

The radius of gyration $R_g$ describes how the scattering mass is spread around the centre. For a uniform
sphere $R_g = \sqrt{3/5}\;r$. A constant such as $3/5$ has to be wrapped in a `DescriptorNumber` before
`es.math` accepts it:

In [12]:
three_fifths = DescriptorNumber(name='three_fifths', value=3 / 5, variance=0.0)

gyration = es.math.sqrt(three_fifths) * model.r
show('radius of gyration', gyration)

radius of gyration = 70.19 ± 0.2 Å


### Exercise 2: The first minimum

The form factor of a sphere has its first minimum where $qr = 4.4934$. Work out the $q$ of the first minimum for the fitted radius, in $\text{Å}^{-1}$, and the length $2\pi/q$ that belongs to it.

In [13]:
radius = model.r

<details class="question">
<summary>Hint 1</summary>

Wrap the constant 4.4934 in a `DescriptorNumber` without uncertainty, then divide it by the radius.

</details>

<details class="question">
<summary>Hint 2</summary>

The unit of the result is $1/\text{Å}$. The length $2\pi/q$ is again in $\text{Å}$.

</details>

In [14]:
%%solution Both results carry the relative uncertainty of the radius.
x_min = DescriptorNumber(name='x_min', value=4.4934, variance=0.0)
q_min = x_min / radius
length = 2 * np.pi / q_min
show('q_min', q_min)
show('2 pi / q_min', length)

## The Guinier approximation

At small $q$ the scattering of any object falls off as $I(q)/I(0) \approx \exp(-q^2 R_g^2/3)$ (the Guinier
approximation). With `es.math.exp`, the value at a given $q$ comes with an uncertainty from $R_g$. The
exponent has to be dimensionless: $q$ in $\text{Å}^{-1}$ times $R_g$ in $\text{Å}$ is:

In [15]:
q_example = DescriptorNumber(name='q_example', value=0.02, unit='1/angstrom', variance=0.0)

guinier = es.math.exp(-((q_example * gyration) ** 2) / 3)
show('Guinier I/I(0)', guinier)

qr = q_example.value * model.r.value
exact = (3 * (np.sin(qr) - qr * np.cos(qr)) / qr**3) ** 2
print(f'sphere I/I(0) = {exact:.4g}')

Guinier I/I(0) = 0.5184 ± 0.002 dimensionless
sphere I/I(0) = 0.501


The approximation is a few per cent above the exact sphere intensity at this $q$, and the difference grows
with $q$:

In [16]:
curve_q = np.linspace(0.002, 0.06, 59)

approximation, exact_curve = [], []
for value in curve_q:
    point = DescriptorNumber(name='q', value=value, unit='1/angstrom', variance=0.0)
    approximation.append(es.math.exp(-((point * gyration) ** 2) / 3).value)
    x = value * model.r.value
    exact_curve.append((3 * (np.sin(x) - x * np.cos(x)) / x**3) ** 2)

fig = go.Figure()
fig.add_trace(go.Scatter(x=curve_q, y=exact_curve, mode='lines', name='sphere'))
fig.add_trace(
    go.Scatter(x=curve_q, y=approximation, mode='lines', name='Guinier', line=dict(dash='dash'))
)
fig.update_layout(
    xaxis_title='q / (1/Å)',
    yaxis_title='I(q) / I(0)',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.65, y=0.95),
)
show_figure(fig)

### Exercise 3: Guinier at another q

Work out the Guinier value $I(q)/I(0)$ at $q = 0.03\ \text{Å}^{-1}$ for the fitted sphere, with its uncertainty. Then print the exact sphere value at the same $q$ for comparison.

In [17]:
q_example = DescriptorNumber(name='q_example', value=0.03, unit='1/angstrom', variance=0.0)
three_fifths = DescriptorNumber(name='three_fifths', value=3 / 5, variance=0.0)

<details class="question">
<summary>Hint 1</summary>

The radius of gyration is `es.math.sqrt(three_fifths) * model.r`, as above.

</details>

<details class="question">
<summary>Hint 2</summary>

Compute the exact value with NumPy from $qr$: it is $\left(3[\sin x - x\cos x]/x^3\right)^2$ with $x = qr$.

</details>

In [18]:
%%solution At this larger $q$ the approximation is further from the exact value.
gyration = es.math.sqrt(three_fifths) * model.r
guinier = es.math.exp(-((q_example * gyration) ** 2) / 3)
x = q_example.value * model.r.value
exact = (3 * (np.sin(x) - x * np.cos(x)) / x**3) ** 2
show('Guinier I/I(0)', guinier)
print(f'sphere I/I(0) = {exact:.4g}')

### Exercise 4: Where the Guinier curve reaches 1/e

The Guinier curve reaches $1/e$ where $q R_g = \sqrt{3}$. Work out that $q$ for the fitted sphere, with its uncertainty, in $\text{Å}^{-1}$.

In [19]:
three = DescriptorNumber(name='three', value=3.0, variance=0.0)
three_fifths = DescriptorNumber(name='three_fifths', value=3 / 5, variance=0.0)

<details class="question">
<summary>Hint 1</summary>

`es.math.sqrt(three)` is the dimensionless number $\sqrt{3}$.

</details>

<details class="question">
<summary>Hint 2</summary>

Divide it by the radius of gyration, `es.math.sqrt(three_fifths) * model.r`.

</details>

In [20]:
%%solution This $q$ is $\sqrt{5}/r$, so its relative uncertainty is that of the radius.
gyration = es.math.sqrt(three_fifths) * model.r
q_one_over_e = es.math.sqrt(three) / gyration
show('q at 1/e', q_one_over_e)

## Good to know

* **The functions of angles need an angle unit.** The product $qr$ is dimensionless, so the full sphere
  amplitude $\sin(qr) - qr\cos(qr)$ cannot be evaluated with `es.math.sin` and `es.math.cos` directly:
  they ask for `rad` or `deg`, while `es.math.sinc` accepts a dimensionless number.

In [21]:
x = q_example * model.r
try:
    es.math.sin(x)
except Exception as exc:
    print(f'{type(exc).__name__}: {exc}')

print(f'sinc(x) = {es.math.sinc(x).value:.4f} for x = {x.value:.3f}')

UnitError: Trigonometric function requires rad or deg unit, got dimensionless.
sinc(x) = 0.1510 for x = 2.719


* **A derived result is a snapshot.** It does not follow the number it came from. The diameter below keeps
  its value when the radius changes afterwards:

In [22]:
radius = Parameter(name='radius', value=90.0, variance=0.3**2, unit='angstrom')
snapshot = 2 * radius
radius.value = 100.0

print(f'radius = {radius.value:.1f} Å, diameter = {snapshot.value:.1f} Å')

radius = 100.0 Å, diameter = 180.0 Å


* **Fitted parameters can be correlated, but arithmetic treats them as independent.** A quantity that depends on several fitted parameters, such as the contrast times the volume, therefore has an uncertainty that cannot be fully trusted. The notebook on correlation shows how large the effect can
  be.